In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 1. Load prepared dataset
data_path = Path(
    "../data/processed/intersection_day_2018_w5_features.csv"
)

model_df = pd.read_csv(
    data_path,
    parse_dates=["prediction_date", "target_date"]
)


# 2. Prepare training data
feature_cols = [
    "historical_ksi_count",
    "historical_traffic_volume",
    "day_of_week",
    "month"
]

train_mask = model_df["prediction_date"] <= "2018-09-29"

X_train = model_df.loc[train_mask, feature_cols]
y_train = model_df.loc[train_mask, "target"]


# 3. Numerical preprocessing
numeric_features = [
    "historical_ksi_count",
    "historical_traffic_volume"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# 4. Categorical preprocessing
categorical_features = ["day_of_week", "month"]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# 5. Complete ML pipeline
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)


# 6. Temporal cross-validation
train_dates = model_df.loc[X_train.index, "prediction_date"]
unique_dates = pd.Index(train_dates.unique()).sort_values()

tscv = TimeSeriesSplit(n_splits=3, gap=1)

cv_splits = []

for train_days_idx, valid_days_idx in tscv.split(unique_dates):
    train_days = unique_dates[train_days_idx]
    valid_days = unique_dates[valid_days_idx]

    train_idx = np.flatnonzero(
        train_dates.isin(train_days).to_numpy()
    )

    valid_idx = np.flatnonzero(
        train_dates.isin(valid_days).to_numpy()
    )

    cv_splits.append((train_idx, valid_idx))


# 7. Essential summary
print("Training observations:", len(X_train))
print("Training positives:", y_train.sum())
print("Temporal CV folds:", len(cv_splits))

for i, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}: "
        f"train positives={y_train.iloc[train_idx].sum()}, "
        f"validation positives={y_train.iloc[valid_idx].sum()}"
    )

Training observations: 613223
Training positives: 81
Temporal CV folds: 3
Fold 1: train positives=14, validation positives=20
Fold 2: train positives=34, validation positives=18
Fold 3: train positives=51, validation positives=29


In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "logisticregression__C": [0.01, 0.1, 1.0, 10.0, 100.0]
}

grid = GridSearchCV(
                    model, 
                    param_grid, 
                    cv=cv_splits, scoring="average_precision", 
                    n_jobs=1, 
                    refit=True
                )

grid.fit(X_train, y_train)

print(grid.best_params_, grid.best_score_)

{'logisticregression__C': 0.1} 0.0006286027681550712


In [5]:
results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "param_logisticregression__C",
        "mean_test_score",
        "std_test_score",
        "split0_test_score",
        "split1_test_score",
        "split2_test_score"
    ]
].sort_values("mean_test_score", ascending=False)

display(results)

,param_logisticregression__C,mean_test_score,std_test_score,split0_test_score,split1_test_score,split2_test_score
1,0.10,0.000629,0.000241,0.000502,0.000418,0.000966
2,1.00,0.000626,0.000242,0.000492,0.000420,0.000966
3,10.00,0.000626,0.000243,0.000491,0.000420,0.000966
4,100.00,0.000625,0.000243,0.000490,0.000420,0.000966
0,0.01,0.000601,0.000349,0.000374,0.000334,0.001094


In [7]:
from sklearn.metrics import average_precision_score

val_mask = model_df["prediction_date"].between(
    "2018-10-01",
    "2018-12-31"
)

X_val = model_df.loc[val_mask, feature_cols]
y_val = model_df.loc[val_mask, "target"]

best_model = grid.best_estimator_

y_val_proba = best_model.predict_proba(X_val)[:, 1]

val_ap = average_precision_score(y_val, y_val_proba)

print(f"Tuned LogisticRegression AP: {val_ap:.6f}")

Tuned LogisticRegression AP: 0.000372


## Experiment Conclusions

Hyperparameter tuning was performed using `GridSearchCV` with three temporal cross-validation folds and five values of the regularization parameter `C`.

The best configuration was `C=0.1`, achieving a mean cross-validation Average Precision of approximately **0.000629**, compared with **0.000626** for the default `C=1.0`.

On the October–December 2018 development validation period, the tuned model achieved **AP = 0.000372**, matching the untuned model to six decimal places. Both outperformed the historical baseline (AP = 0.000182).

**Conclusion:** Hyperparameter tuning did not produce a meaningful improvement. The small CV difference is insufficient to demonstrate better generalization, especially given the limited number of positive observations.

All tuning was performed within the training data using temporal CV. The final test set was not used. The results remain exploratory and require independent future evaluation.